# Exploration CNPS : salaires et panel entreprise

Statistiques descriptives sur `cnps_cleaned.parquet` (bucket `gold`), puis jointure avec le référentiel
ANSTAT sur `RAISON_SOCIALE` normalisée (~96.5% de correspondance, cf. `src/cnps/05_1_jointure_anstat.py`
pour la version pipeline de cette jointure — ce notebook sert à l'explorer, pas à la reproduire en prod).

In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path.cwd() / "src"))
from cnps.config import load_config
from cnps.storage import read_parquet, read_excel_bytes

cfg = load_config()
minio_cfg = cfg.minio
CLEANED_OBJECT = f"{minio_cfg.cleaned_prefix}cnps_cleaned.parquet"
ANSTAT_OBJECT = "cnps/REQUETES_ANSTAT_MODULE_EMPLOYEURS.xlsx"

pl.Config.set_tbl_rows(30)

## 1. Chargement

In [ ]:
df = read_parquet(minio_cfg, minio_cfg.cleaned_bucket, CLEANED_OBJECT)
print(f"{df.height:,} lignes, {df.width} colonnes")
df.head(5)

In [ ]:
# ANSTAT est un fichier local depose sur MinIO (staging/cnps/), pas un output du pipeline
df_anstat = pl.read_excel(read_excel_bytes(minio_cfg, minio_cfg.raw_bucket, ANSTAT_OBJECT), sheet_name="DATA")
print(f"{df_anstat.height:,} lignes, {df_anstat.width} colonnes")
df_anstat.head(5)

## 2. Salaires

Colonnes confirmées : `SALAIRE_BRUT_MENS`, `AGE_EMPLOYE`, `EFFECTIF_SALARIES`, `ANCIENNETE_ENTREPRISE`,
`SEXE`, `SITUATION_MATRIMONIALE`, `NIVEAU_ETUDE`, `TYPE_SALARIE`, `STATUT_TRAVAILLEUR`,
`CATEGORIE_ENTREPRISE`.

In [ ]:
df.select("SALAIRE_BRUT_MENS", "AGE_EMPLOYE", "EFFECTIF_SALARIES", "ANCIENNETE_ENTREPRISE").describe()

In [ ]:
df.select(
    pl.col("SALAIRE_BRUT_MENS").quantile(q).alias(f"p{int(q*100)}")
    for q in [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

In [ ]:
for col in ["SEXE", "SITUATION_MATRIMONIALE", "NIVEAU_ETUDE", "TYPE_SALARIE", "STATUT_TRAVAILLEUR", "CATEGORIE_ENTREPRISE"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().sort("count", descending=True).head(8))
    print()

In [ ]:
df.group_by("PERIOD").agg(
    pl.len().alias("n_declarations"),
    pl.col("ID_INDIV").n_unique().alias("n_individus"),
    pl.col("ID_EMPLOYEUR").n_unique().alias("n_entreprises"),
    pl.col("SALAIRE_BRUT_MENS").mean().round(0).alias("salaire_moyen"),
).sort("PERIOD")

## 3. Secteur d'activité (nomenclature CNPS)

In [ ]:
df.group_by("SECTEUR_ACTIVITE").agg(
    pl.len().alias("n_lignes"),
    pl.col("ID_EMPLOYEUR").n_unique().alias("n_entreprises"),
    pl.col("SALAIRE_BRUT_MENS").mean().round(0).alias("salaire_moyen"),
    pl.col("SALAIRE_BRUT_MENS").median().round(0).alias("salaire_median"),
).sort("n_lignes", descending=True)

## 4. Jointure CNPS ↔ ANSTAT sur `RAISON_SOCIALE`

Aucun identifiant commun entre les deux sources (pas de RCCM/DFE côté CNPS) : seule la raison sociale,
normalisée, sert de clé. Le référentiel ANSTAT a plusieurs enregistrements par entreprise dans le temps
(changements de secteur/motif) — on garde le plus récent avant de joindre, sinon la jointure duplique des
lignes CNPS.

In [ ]:
def normalize_raison_sociale(col: pl.Expr) -> pl.Expr:
    return (
        col.str.to_uppercase()
        .str.replace_all(r"[^A-Z0-9 ]", "")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )

cnps_firms = (
    df.select("ID_EMPLOYEUR", "RAISON_SOCIALE")
    .unique(subset="ID_EMPLOYEUR", keep="first")
    .with_columns(normalize_raison_sociale(pl.col("RAISON_SOCIALE")).alias("RS_NORM"))
)

anstat_firms = (
    df_anstat.sort("DATE DE DEBUT D'ACTIVITE", descending=True, nulls_last=True)
    .with_columns(normalize_raison_sociale(pl.col("RAISON_SOCIALE")).alias("RS_NORM"))
    .unique(subset="RS_NORM", keep="first")
)

print("Entreprises CNPS  :", cnps_firms.height)
print("Entreprises ANSTAT:", anstat_firms.height, "(apres deduplication sur raison sociale normalisee)")

In [ ]:
matched = cnps_firms.join(anstat_firms, on="RS_NORM", how="left", suffix="_anstat")

n_matched = matched.filter(pl.col("SECTEUR_ACTIVITE_anstat").is_not_null()).height
n_total = matched.height
print(f"Correspondance ANSTAT : {n_matched:,} / {n_total:,} ({n_matched / n_total * 100:.1f}%)")

matched.filter(pl.col("SECTEUR_ACTIVITE_anstat").is_not_null()).select(
    "RAISON_SOCIALE", "SECTEUR_ACTIVITE_anstat", "FORME JURIDIQUE", "NUMERO_RCCM"
).head(10)

## 5. Salaires par secteur ANSTAT (CEPICI)

On reporte le secteur ANSTAT sur chaque ligne salariale via `ID_EMPLOYEUR`, puis on compare aux
statistiques par secteur CNPS (section 3).

In [ ]:
secteur_anstat_map = matched.select("ID_EMPLOYEUR", pl.col("SECTEUR_ACTIVITE_anstat").alias("SECTEUR_ANSTAT"))
df_enrichi = df.join(secteur_anstat_map, on="ID_EMPLOYEUR", how="left")

pct_connu = round(df_enrichi["SECTEUR_ANSTAT"].is_not_null().sum() / df_enrichi.height * 100, 1)
print(f"Lignes salariales avec secteur ANSTAT connu : {pct_connu}%")

df_enrichi.filter(pl.col("SECTEUR_ANSTAT").is_not_null()).group_by("SECTEUR_ANSTAT").agg(
    pl.len().alias("n_lignes"),
    pl.col("ID_EMPLOYEUR").n_unique().alias("n_entreprises"),
    pl.col("SALAIRE_BRUT_MENS").mean().round(0).alias("salaire_moyen"),
    pl.col("SALAIRE_BRUT_MENS").median().round(0).alias("salaire_median"),
).sort("salaire_moyen", descending=True)

## 6. Synthèse

- **Salaires** : distribution asymétrique typique (médiane très inférieure à la moyenne), cohérente avec
  un marché salarial où une minorité de hauts salaires tire la moyenne vers le haut — voir percentiles
  section 2.
- **Panel entreprise** : `ID_EMPLOYEUR` est la clé fiable (67 810 entreprises), couplée à `RAISON_SOCIALE`
  à 100%. Le panel équilibré avec non-déclarants (`D_JT`) est construit par le pipeline
  (`05_base_entreprises.py`), pas reproduit ici.
- **Jointure ANSTAT** : ~96.5% de correspondance sur raison sociale normalisée. Le référentiel ANSTAT a
  plusieurs enregistrements historiques par entreprise — dédupliquer sur le plus récent est nécessaire
  avant toute jointure, sinon les lignes CNPS se dupliquent silencieusement.
- **Suite** : la jointure est maintenant industrialisée dans le pipeline
  (`src/cnps/05_1_jointure_anstat.py`, étape `JOINTURE_ANSTAT`), qui écrit directement les colonnes
  `SECTEUR_ACTIVITE_ANSTAT`, `FORME_JURIDIQUE_ANSTAT`, `NUMERO_RCCM`, `NUMERO_DFE` dans `firm_base.parquet`.
  Ce notebook explore la donnée ; il ne remplace pas cette étape.